# 4-Stream Face Recognition — InsightFace SCRFD + W600K on TensorRT + NVDEC (NVIDIA GPU only)

```
Stream 1..4 (files now, cameras later)
        ↓
NVDEC GPU Video Decode
        ↓
InsightFace SCRFD Face Detection (TensorRT)
        ↓
Face Crop
        ↓
W600K / w600k_r50 (112 × 112)
        ↓
512-D Face Embedding
        ↓
Persistent Gallery Database
        ↓
Cosine Similarity
        ↓
60% or higher → Known    |    below 60% → Unknown / ALERT
```

**Requirements:** Runtime → Change runtime type → **NVIDIA GPU** (e.g. T4/L4/A100).
Video decoding is performed through **FFmpeg NVDEC (CUDA hardware decode)**. Face detection and
face embedding run as **TensorRT CUDA engines**. CPU is used for frame transfer, face cropping,
drawing, output writing and the lightweight IoU tracker.


## 0) GPU check (hard requirement)

In [ ]:
import subprocess, torch

def require_nvidia_gpu():
    """Stop the notebook unless an NVIDIA CUDA GPU is available. No CPU fallback."""
    ok = torch.cuda.is_available() and torch.version.cuda is not None
    if ok:
        try:
            smi = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10)
            ok = smi.returncode == 0 and "GPU" in smi.stdout
        except Exception:
            ok = False
    if not ok:
        msg = "NVIDIA CUDA GPU is required for this pipeline."
        print(msg)
        raise RuntimeError(msg)
    p = torch.cuda.get_device_properties(0)
    print(f"GPU OK: {p.name} | {p.total_memory/1e9:.1f} GB | CUDA {torch.version.cuda}")

require_nvidia_gpu()

## 1) Install dependencies & download models

In [ ]:
# TensorRT: GPU inference runtime
# ONNX: used by TensorRT to build the SCRFD and W600K engines
# OpenCV: image operations/cropping/writing; video pixels are decoded by FFmpeg NVDEC
!pip -q install tensorrt onnx onnxsim opencv-python-headless

# InsightFace buffalo_l pack contains BOTH:
#   buffalo_l/det_10g.onnx      -> InsightFace SCRFD detector
#   buffalo_l/w600k_r50.onnx    -> W600K face-recognition embedder
!wget -q -nc https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip
!unzip -q -n buffalo_l.zip -d buffalo_l

# CuPy is kept because the project already uses it for the existing large-array box-rescale path.
!pip -q install cupy-cuda12x 2>/dev/null || echo "cupy-cuda12x install failed -- NumPy path will be used"


## 2) Imports & configuration

In [ ]:
import os, glob, zipfile, base64, time, queue, threading
from concurrent.futures import ThreadPoolExecutor
from abc import ABC, abstractmethod
from collections import Counter

import cv2
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
import tensorrt as trt
import subprocess # Added subprocess import

# ---------- CuPy (optional GPU array backend -- Step 1) ----------
# Used ONLY for the specific vectorized array math identified in the audit as a genuine, safe
# win (see rescale_boxes_gpu below). Everything else that is numerically heavy in this pipeline
# (SCRFD box decoding, NMS, cosine similarity) is ALREADY implemented as CUDA torch tensor ops, so
# it does not need CuPy -- adding it there would just insert a torch<->CuPy conversion for no
# benefit. CPU NumPy stays wherever the data is CPU-side already and would gain nothing from a
# round trip to the GPU (e.g. cv2 frame arrays, the tiny per-gallery-update mean-embedding matrix).
try:
    import cupy as cp
    _ = cp.zeros(1) + 1        # touch the GPU once so import-only "success" isn't reported falsely
    CUPY_AVAILABLE = True
except Exception as _cupy_err:
    cp = None
    CUPY_AVAILABLE = False
USE_CUPY = CUPY_AVAILABLE   # flip to False to force the NumPy path even if CuPy imported fine

# require_nvidia_gpu() is defined in the GPU-check cell at the top of the notebook.

require_nvidia_gpu()                 # re-check after imports
DEVICE = torch.device("cuda:0")      # ALL inference tensors live on this device

# ---------- Pipeline settings ----------
MAX_STREAMS   = 4        # pipeline is optimized for up to 4 input streams
THRESHOLD     = 0.60     # cosine similarity >= 60% -> Known   (unchanged)
DET_SIZE      = 640      # InsightFace SCRFD det_10g input is letterboxed to 640 x 640
MIN_DET_SCORE = 0.8      # ignore SCRFD detections below this confidence (unchanged)
USE_FP16      = True     # build TensorRT engines with FP16 kernels (I/O stays FP32)

# ---------- Engine batch ranges (min, opt, max) ----------
DET_BATCH = (1, 1, 1)              # SCRFD engine is fixed batch 1 (decode_scrfd handles one image per call)
EMB_BATCH = (1, 16, 64)            # W600K: all faces from all streams in one call

# ---------- Files ----------
SCRFD_ONNX   = "buffalo_l/det_10g.onnx"
SCRFD_ENGINE = f"scrfd_det10g_{DET_SIZE}_b{DET_BATCH[2]}.engine"
W600K_ONNX  = "buffalo_l/w600k_r50.onnx"
W600K_ENGINE = f"w600k_r50_b{EMB_BATCH[2]}.engine"

# ---------- Video frame sampling ----------
# Maximum number of frames PER SECOND OF VIDEO that will enter the AI pipeline.
#   input FPS >  TARGET_FPS -> frames are skipped BEFORE detection so only ~TARGET_FPS/s are processed
#   input FPS <= TARGET_FPS -> every frame is processed (1 FPS ... 30 FPS all work)
TARGET_FPS = 30

# ---------- Persistent gallery ----------
# Gallery images and their embeddings live here and are NEVER deleted by a new upload.
# /content survives for the whole Colab session. To survive a runtime restart too, mount Drive
# (see the Drive cell in section 5) and use e.g. "/content/drive/MyDrive/face_recognition_project" as PROJECT_DIR.
PROJECT_DIR = "/content/face_recognition_project"
GALLERY_DIR = f"{PROJECT_DIR}/gallery"                   # permanent reference images
GALLERY_DB  = f"{PROJECT_DIR}/gallery_embeddings.npz"    # persistent per-image embedding database


# ---------- Adaptive frame processing (NEW) ----------
# Detection does NOT have to run on every processed frame: a lightweight IoU tracker (added below,
# no new dependency) carries face boxes forward between detection frames, and recognition (W600K) is
# cached per track ID instead of being re-run every frame. These knobs control how aggressively.
#
#   DETECTION_INTERVAL   : run SCRFD every Nth processed frame. Between detection frames the tracker
#                          reuses the last known box for each track (no motion model / optical flow,
#                          so boxes lag behind fast-moving faces the larger this is set).
#   RECOGNITION_INTERVAL : re-run W600K for an already-identified, stable track every N processed
#                          frames. A track is ALSO re-recognized immediately, regardless of this
#                          interval, when it is brand-new or its cached identity is low-confidence.
#   MIN_RECOG_SCORE      : cached identity is treated as "uncertain" (forces re-recognition every
#                          frame) if its cosine similarity is below this. Defaults to THRESHOLD.
#   TRACK_IOU_THRESHOLD  : minimum IoU to match a new detection to an existing track.
#   TRACK_MAX_MISSES     : consecutive detection rounds a track can go unmatched before it is dropped.
MIN_RECOG_SCORE       = THRESHOLD
TRACK_IOU_THRESHOLD   = 0.3
TRACK_MAX_MISSES      = 10

# ---------- Processing-mode presets (Step 9) ----------
# Change ACTIVE_MODE and re-run this cell (or just reassign DETECTION_INTERVAL / RECOGNITION_INTERVAL
# directly below -- they are plain variables, nothing else in the notebook hard-codes them).
PROCESSING_MODES = {
    "HIGH_ACCURACY": dict(DETECTION_INTERVAL=1, RECOGNITION_INTERVAL=5),
    "BALANCED":       dict(DETECTION_INTERVAL=3, RECOGNITION_INTERVAL=15),
    "HIGH_SPEED":     dict(DETECTION_INTERVAL=5, RECOGNITION_INTERVAL=20),
}
ACTIVE_MODE = "HIGH_SPEED"
DETECTION_INTERVAL   = PROCESSING_MODES[ACTIVE_MODE]["DETECTION_INTERVAL"]
RECOGNITION_INTERVAL = PROCESSING_MODES[ACTIVE_MODE]["RECOGNITION_INTERVAL"]
print(f"Processing mode: {ACTIVE_MODE}  "
      f"(DETECTION_INTERVAL={DETECTION_INTERVAL}, RECOGNITION_INTERVAL={RECOGNITION_INTERVAL})")

# ---------- Batch video queue (Step 12) ----------
MAX_VIDEOS_PER_BATCH = 30   # hard cap: videos are processed ONE AT A TIME (sequential queue), never
                            # concurrently -- avoids overloading Colab GPU/RAM regardless of how many
                            # of the 1..30 uploaded videos are queued.

# ---------- Frame-chunked batching (NEW -- this is the main real-time speed fix) ----------
# TensorRT has a fixed per-call overhead (kernel launch, context/shape setup) that is paid EVERY
# time .predict()/.infer() is invoked, almost regardless of batch size. Calling SCRFD/W600K once
# per individual frame (as before) pays that overhead per frame even though DETECTION_INTERVAL /
# RECOGNITION_INTERVAL already reduce how many frames need inference. Buffering FRAME_CHUNK_SIZE
# frames and batching every detection-due (and recognition-due) frame within the chunk into ONE
# TensorRT call amortizes that fixed cost over many frames instead of paying it every time.
# Larger chunks = fewer, bigger TRT calls = closer to real-time, at the cost of more frames held
# in host RAM at once (chunk_size BGR frames -- fine at typical resolutions).
FRAME_CHUNK_SIZE = 32

# ---------- Async decode/GPU overlap (NEW) ----------
# True: a background thread decodes the NEXT frame chunk while the GPU (detection/recognition)
# and tracker/writer work through the CURRENT chunk (see process_single_video). This only removes
# idle waiting -- it does not change what is computed or the output frame order -- so it is safe
# to leave on. Set False to fall back to the original fully-serial loop for debugging/comparison.
USE_ASYNC_PIPELINE = True

# ---------- NVDEC (GPU hardware video decode) -- Steps 2/3 ----------
# True: video pixels are decoded by FFmpeg with CUDA/NVDEC.  The reader keeps the same
# frame-sampling/output logic as before; skipped source frames are read and discarded.
USE_NVDEC = True

# ---------- Cross-video pipelining (NEW -- Steps 4/6) ----------
# While video N is being decoded + run through TensorRT (GPU-bound), the H.264 re-encode of the
# PREVIOUS video (an external ffmpeg process -- CPU libx264 threads, or the GPU's separate NVENC
# hardware block) can run at the same time without contending with TensorRT's CUDA compute.
# MAX_CONCURRENT_ENCODES caps this at 1 background encode job so at most 2 videos are ever
# "in flight" (one on GPU compute, one encoding) -- never one thread per uploaded video.
MAX_CONCURRENT_ENCODES = 1

# ---------- Concurrent multi-stream mode (NEW) ----------
# N videos are decoded at once (one reader thread + bounded buffer each) but share ONE GPU worker that
# batches the latest frame of every stream into a single TensorRT call. Extra uploads wait in the queue.
CONCURRENT_STREAMS = 4     # videos processed simultaneously (1 = old sequential behaviour); 2-6 sensible
STREAM_QUEUE_DEPTH = 32    # max decoded frames buffered per stream (bounds RAM)
LIVE_MODE = False          # False: files -- every frame is written, only inference is skipped.
                           # True : live feeds -- drop OLDEST buffered frames when the GPU falls behind.
ALERT_UNKNOWN_HITS = 2     # consecutive "Unknown" recognitions before an ALERT fires (cuts false alarms)


## 3) TensorRT layer (engine builder + runtime)

In [ ]:
TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

_TRT2TORCH = {trt.DataType.FLOAT: torch.float32,
              trt.DataType.HALF:  torch.float16,
              trt.DataType.INT32: torch.int32}
for _n, _t in (("INT64", torch.int64), ("INT8", torch.int8), ("BOOL", torch.bool)):
    if hasattr(trt.DataType, _n):          # only present in some TensorRT versions
        _TRT2TORCH[getattr(trt.DataType, _n)] = _t


def build_engine(onnx_path, engine_path, min_shape, opt_shape, max_shape, fp16=True):
    """
    ONNX -> serialized TensorRT engine with a dynamic-batch optimization profile.
    Skipped if `engine_path` already exists (engines are GPU/TensorRT-version specific,
    so delete the .engine file if you change GPU or TensorRT version).
    """
    if os.path.exists(engine_path):
        print("engine cached:", engine_path); return engine_path
    print("building TensorRT engine (this can take several minutes)...", engine_path)

    builder = trt.Builder(TRT_LOGGER)
    try:    # TensorRT < 10 needs the explicit-batch flag; >= 10 removed/ignores it
        network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    except AttributeError:
        network = builder.create_network(0)

    parser = trt.OnnxParser(network, TRT_LOGGER)
    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors): print(parser.get_error(i))
            raise RuntimeError(f"TensorRT could not parse {onnx_path}")

    print(f"TensorRT network has {network.num_outputs} outputs after parsing ONNX for {os.path.basename(onnx_path)}.") # Debug print

    # ADDED DIAGNOSTIC PRINTOUTS HERE
    print("--- Network Inputs ---")
    for i in range(network.num_inputs):
        inp = network.get_input(i)
        print(f"  Input {i}: Name={inp.name}, Shape={inp.shape}, DType={inp.dtype}")
    print("--- Network Outputs ---")
    for i in range(network.num_outputs):
        out = network.get_output(i)
        print(f"  Output {i}: Name={out.name}, Shape={out.shape}, DType={out.dtype}")
    # END ADDED DIAGNOSTIC PRINTOUTS

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 2 << 30)      # 2 GB scratch
    if fp16:
        # `BuilderFlag.FP16` is deprecated/removed in the newest TensorRT (strongly-typed networks):
        # there the precision comes from the ONNX itself, so just skip the flag.
        if hasattr(trt.BuilderFlag, "FP16"):
            config.set_flag(trt.BuilderFlag.FP16)
        else:
            print("Note: this TensorRT version has no FP16 builder flag; using the precision stored in the ONNX.")

    profile = builder.create_optimization_profile()                          # dynamic batch range
    profile.set_shape(network.get_input(0).name, min_shape, opt_shape, max_shape)
    config.add_optimization_profile(profile)

    blob = builder.build_serialized_network(network, config)
    if blob is None: raise RuntimeError("TensorRT engine build failed")
    with open(engine_path, "wb") as f: f.write(blob)
    return engine_path


class TRTEngine:
    """
    Minimal TensorRT runner operating directly on CUDA torch tensors
    (zero host copies: TensorRT reads/writes torch's GPU memory).
    """
    def __init__(self, engine_path):
        require_nvidia_gpu()
        runtime = trt.Runtime(TRT_LOGGER)
        with open(engine_path, "rb") as f:
            self.engine = runtime.deserialize_cuda_engine(f.read())
        if self.engine is None:
            raise RuntimeError(f"Could not load TensorRT engine {engine_path!r}. Engines are specific to the "
                               f"GPU + TensorRT version: delete the file and re-run the build cell.")
        self.context = self.engine.create_execution_context()
        names = [self.engine.get_tensor_name(i) for i in range(self.engine.num_io_tensors)]
        self.input_name  = [n for n in names if self.engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT][0]
        self.output_names = [n for n in names if self.engine.get_tensor_mode(n) == trt.TensorIOMode.OUTPUT]

    def infer(self, x):
        """x: CUDA float tensor (B,C,H,W). Returns {output_name: CUDA tensor}."""
        assert x.is_cuda, "TensorRT input must be a CUDA tensor"
        x = x.contiguous().float()
        self.context.set_input_shape(self.input_name, tuple(x.shape))        # pick batch size
        self.context.set_tensor_address(self.input_name, x.data_ptr())
        outs = {}
        for n in self.output_names:
            shape = tuple(self.context.get_tensor_shape(n))
            outs[n] = torch.empty(shape, dtype=_TRT2TORCH[self.engine.get_tensor_dtype(n)], device=x.device)
            self.context.set_tensor_address(n, outs[n].data_ptr())
        stream = torch.cuda.current_stream()
        if not self.context.execute_async_v3(stream.cuda_stream):            # GPU batch inference
            raise RuntimeError("TensorRT execution failed")
        stream.synchronize()
        return outs

## 4) Model Interface (SCRFD and W600K on top of TensorRT)

In [ ]:
# ============================================================
# Model Interface
# ============================================================

class ModelInterface(ABC):
    """Common interface: raw inputs in, results out."""

    @abstractmethod
    def predict(self, inputs):
        ...


# ============================================================
# InsightFace SCRFD det_10g configuration
#
# TensorRT engine:
#   3 score outputs
#   3 bbox outputs
#   3 landmark outputs
#
# Feature strides:
#   8, 16, 32
# ============================================================

SCRFD_STRIDES = [8, 16, 32]
SCRFD_NMS_IOU = 0.4

MEAN_SCRFD = 127.5
STD_SCRFD = 128.0


# ============================================================
# SCRFD preprocessing
# ============================================================

def letterbox_scrfd(img, size):
    """Preserve aspect ratio and pad image to size x size."""

    h, w = img.shape[:2]

    im_ratio = float(h) / float(w)

    if im_ratio > 1.0:
        new_h = size
        new_w = max(1, int(new_h / im_ratio))
    else:
        new_w = size
        new_h = max(1, int(new_w * im_ratio))

    scale = float(new_h) / float(h)

    resized = cv2.resize(
        img,
        (new_w, new_h)
    )

    canvas = np.zeros(
        (size, size, 3),
        dtype=np.uint8
    )

    canvas[:new_h, :new_w] = resized

    return canvas, scale


# ============================================================
# SCRFD output decoder
#
# IMPORTANT:
# TensorRT outputs are flattened:
#
# score:
#   [12800, 1]
#   [3200, 1]
#   [800, 1]
#
# bbox:
#   [12800, 4]
#   [3200, 4]
#   [800, 4]
#
# Therefore:
#   DO NOT use output.shape[0] as batch size.
#
# Current SCRFD TensorRT engine uses batch = 1.
# ============================================================

def decode_scrfd(
    outs,
    input_size,
    score_thr,
    nms_iou=SCRFD_NMS_IOU
):

    tensors = []

    # --------------------------------------------------------
    # Convert TensorRT outputs to PyTorch tensors
    # --------------------------------------------------------

    for out in outs:

        if isinstance(out, torch.Tensor):
            t = out

        else:
            t = torch.from_numpy(
                np.asarray(out)
            )

        if t.device != DEVICE:
            t = t.to(DEVICE)

        tensors.append(t)

    # --------------------------------------------------------
    # Separate:
    #   score     -> 1
    #   bbox      -> 4
    #   landmark  -> 10
    # --------------------------------------------------------

    score_out = []
    bbox_out = []
    landmark_out = []

    for t in tensors:

        if t.ndim == 1:
            last_dim = 1
        else:
            last_dim = t.shape[-1]

        if last_dim == 1:
            score_out.append(t)

        elif last_dim == 4:
            bbox_out.append(t)

        elif last_dim == 10:
            landmark_out.append(t)

    # --------------------------------------------------------
    # Validate SCRFD outputs
    # --------------------------------------------------------

    if len(score_out) != 3:
        raise RuntimeError(
            f"SCRFD expected 3 score outputs, "
            f"but found {len(score_out)}"
        )

    if len(bbox_out) != 3:
        raise RuntimeError(
            f"SCRFD expected 3 bbox outputs, "
            f"but found {len(bbox_out)}"
        )

    # --------------------------------------------------------
    # Largest -> smallest feature level
    #
    # 12800 -> stride 8
    # 3200  -> stride 16
    # 800   -> stride 32
    # --------------------------------------------------------

    score_out.sort(
        key=lambda x: x.numel(),
        reverse=True
    )

    bbox_out.sort(
        key=lambda x: x.numel(),
        reverse=True
    )

    boxes_all = []
    scores_all = []

    # ========================================================
    # Decode each SCRFD feature level
    # ========================================================

    for idx, stride in enumerate(SCRFD_STRIDES):

        score = score_out[idx]
        bbox = bbox_out[idx]

        # ----------------------------------------------------
        # Flatten score
        # ----------------------------------------------------

        score = score.reshape(-1)

        # ----------------------------------------------------
        # Flatten bbox
        # ----------------------------------------------------

        if bbox.numel() % 4 != 0:
            raise RuntimeError(
                f"Invalid SCRFD bbox output "
                f"at stride {stride}: {bbox.shape}"
            )

        bbox = bbox.reshape(-1, 4)

        # ----------------------------------------------------
        # Candidate count
        # ----------------------------------------------------

        num_candidates = min(
            score.shape[0],
            bbox.shape[0]
        )

        score = score[:num_candidates]
        bbox = bbox[:num_candidates]

        # ----------------------------------------------------
        # Feature-map size
        # ----------------------------------------------------

        feature_h = input_size // stride
        feature_w = input_size // stride

        spatial_locations = (
            feature_h * feature_w
        )

        # ----------------------------------------------------
        # Determine number of anchors
        # ----------------------------------------------------

        if num_candidates % spatial_locations != 0:

            raise RuntimeError(
                f"SCRFD output mismatch at stride "
                f"{stride}: {num_candidates} candidates "
                f"cannot fit {feature_h}x{feature_w}."
            )

        num_anchors = (
            num_candidates //
            spatial_locations
        )

        # ----------------------------------------------------
        # Create grid centers
        # ----------------------------------------------------

        yy, xx = torch.meshgrid(
            torch.arange(
                feature_h,
                device=score.device,
                dtype=torch.float32
            ),
            torch.arange(
                feature_w,
                device=score.device,
                dtype=torch.float32
            ),
            indexing="ij"
        )

        centers = torch.stack(
            [
                xx.reshape(-1),
                yy.reshape(-1)
            ],
            dim=1
        )

        # ----------------------------------------------------
        # Repeat centers for anchors
        # ----------------------------------------------------

        centers = centers.repeat_interleave(
            num_anchors,
            dim=0
        )

        centers = centers[:num_candidates]

        centers = centers * float(stride)

        # ----------------------------------------------------
        # Decode bounding boxes
        # ----------------------------------------------------

        x1 = (
            centers[:, 0]
            - bbox[:, 0] * float(stride)
        )

        y1 = (
            centers[:, 1]
            - bbox[:, 1] * float(stride)
        )

        x2 = (
            centers[:, 0]
            + bbox[:, 2] * float(stride)
        )

        y2 = (
            centers[:, 1]
            + bbox[:, 3] * float(stride)
        )

        boxes = torch.stack(
            [
                x1,
                y1,
                x2,
                y2
            ],
            dim=1
        )

        # ----------------------------------------------------
        # Confidence threshold
        # ----------------------------------------------------

        keep = score >= score_thr

        if torch.any(keep):

            boxes_all.append(
                boxes[keep]
            )

            scores_all.append(
                score[keep]
            )

    # ========================================================
    # No detections
    # ========================================================

    if not boxes_all:

        return [
            np.zeros(
                (0, 5),
                dtype=np.float32
            )
        ]

    # ========================================================
    # Combine all feature levels
    # ========================================================

    boxes = torch.cat(
        boxes_all,
        dim=0
    )

    scores = torch.cat(
        scores_all,
        dim=0
    )

    # ========================================================
    # Clip boxes
    # ========================================================

    boxes[:, 0].clamp_(0, input_size)
    boxes[:, 1].clamp_(0, input_size)
    boxes[:, 2].clamp_(0, input_size)
    boxes[:, 3].clamp_(0, input_size)

    # ========================================================
    # Non-Maximum Suppression
    # ========================================================

    keep = torchvision.ops.nms(
        boxes,
        scores,
        nms_iou
    )

    boxes = boxes[keep]
    scores = scores[keep]

    # ========================================================
    # Final detection format:
    #
    # [x1, y1, x2, y2, confidence]
    # ========================================================

    detections = torch.cat(
        [
            boxes,
            scores.unsqueeze(1)
        ],
        dim=1
    )

    detections = (
        detections
        .detach()
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    return [detections]


# ============================================================
# Rescale SCRFD boxes
# ============================================================

def rescale_boxes_gpu(
    per_frame_dets,
    scales
):

    counts = [
        d.shape[0]
        for d in per_frame_dets
    ]

    total = sum(counts)

    if total == 0:
        return per_frame_dets

    combined = np.concatenate(
        per_frame_dets,
        axis=0
    )

    xp = (
        cp
        if USE_CUPY and total >= 64
        else np
    )

    if xp is cp:
        stacked = cp.asarray(combined)
    else:
        stacked = combined

    scale_per_box = xp.asarray(
        np.repeat(
            np.asarray(scales, np.float32),
            counts
        )
    )

    stacked[:, :4] /= (
        scale_per_box[:, None]
    )

    if xp is cp:
        stacked = cp.asnumpy(stacked)

    out = []

    pos = 0

    for n in counts:

        out.append(
            stacked[pos:pos + n]
        )

        pos += n

    return out


# ============================================================
# SCRFD TensorRT Model
# ============================================================

class SCRFDModel(ModelInterface):
    """
    InsightFace SCRFD TensorRT detector.

    Current TensorRT engine:
        Fixed batch size = 1

    Input:
        List of BGR OpenCV frames

    Output:
        One detection list per frame
    """

    def __init__(
        self,
        engine_path,
        size=DET_SIZE,
        max_batch=1
    ):

        if max_batch != 1:
            raise ValueError("decode_scrfd() decodes one image per TensorRT call; max_batch must be 1")

        self.trt = TRTEngine(
            engine_path
        )

        self.size = size
        self.max_batch = max_batch

        self.mean = MEAN_SCRFD
        self.std = STD_SCRFD

    def predict(self, frames):

        out = []

        # ----------------------------------------------------
        # Current engine uses batch = 1
        # ----------------------------------------------------

        for i in range(
            0,
            len(frames),
            self.max_batch
        ):

            chunk = frames[
                i:i + self.max_batch
            ]

            # ------------------------------------------------
            # Letterbox
            # ------------------------------------------------

            lb = [
                letterbox_scrfd(
                    f,
                    self.size
                )
                for f in chunk
            ]

            # ------------------------------------------------
            # Create NumPy input
            # ------------------------------------------------

            arr = np.stack(
                [
                    c
                    for c, _ in lb
                ]
            ).astype(
                np.float32
            )

            # ------------------------------------------------
            # CPU -> GPU
            # ------------------------------------------------

            x = torch.from_numpy(
                arr
            ).to(
                DEVICE,
                non_blocking=True
            )

            # ------------------------------------------------
            # BGR -> RGB
            # NHWC -> NCHW
            # ------------------------------------------------

            x = (
                x[..., [2, 1, 0]]
                .permute(0, 3, 1, 2)
                .contiguous()
            )

            # ------------------------------------------------
            # SCRFD normalization
            # ------------------------------------------------

            x = (
                x - self.mean
            ) / self.std

            # ------------------------------------------------
            # TensorRT inference
            # ------------------------------------------------

            raw = self.trt.infer(x)

            # ------------------------------------------------
            # Decode SCRFD outputs
            # ------------------------------------------------

            per = decode_scrfd(
                list(raw.values()),
                self.size,
                MIN_DET_SCORE
            )

            # ------------------------------------------------
            # Rescale boxes
            # ------------------------------------------------

            scales = [
                s
                for _, s in lb
            ]

            rescaled = rescale_boxes_gpu(
                per,
                scales
            )

            # ------------------------------------------------
            # Store detections
            # ------------------------------------------------

            for d in rescaled:

                out.append(
                    [
                        tuple(row)
                        for row in d
                    ]
                )

        return out


# ============================================================
# W600K / w600k_r50 configuration
# ============================================================

# W600K is designed for 112x112 input
W600K_INPUT_SIZE = 112

# W600K preprocessing often uses the same normalization as SCRFD
# from the InsightFace buffalo_l pack
MEAN_W600K = MEAN_SCRFD # 127.5
STD_W600K = STD_SCRFD   # 128.0


class W600KModel(ModelInterface):
    """
    W600K / w600k_r50 face recognition embedder.

    Input:
        List of BGR OpenCV face crops (112x112)

    Output:
        512-D face embedding
    """

    def __init__(self, engine_path, size=W600K_INPUT_SIZE):
        self.trt = TRTEngine(engine_path)
        self.size = size
        self.mean = MEAN_W600K
        self.std = STD_W600K

    def predict(self, crops):
        """
        Processes a list of face crops to generate embeddings.
        Input: list of NumPy arrays (H, W, C)
        Output: torch.Tensor (B, 512), L2-normalised so that a dot product == cosine similarity
        """
        if not crops:
            return torch.empty(0, 512, device=DEVICE)

        processed_crops = []
        for crop in crops:
            # Resize to W600K_INPUT_SIZE if not already that size
            if crop.shape[0] != self.size or crop.shape[1] != self.size:
                crop = cv2.resize(crop, (self.size, self.size))

            # Convert to float32 and stack
            processed_crops.append(crop.astype(np.float32))

        # Stack all crops into a single batch
        arr = np.stack(processed_crops)

        # CPU -> GPU
        x = torch.from_numpy(arr).to(DEVICE, non_blocking=True)

        # BGR -> RGB
        # NHWC -> NCHW
        x = (
            x[..., [2, 1, 0]]
            .permute(0, 3, 1, 2)
            .contiguous()
        )

        # Normalization
        x = (x - self.mean) / self.std

        # TensorRT inference
        # W600K typically has a single output (embedding vector)
        raw = self.trt.infer(x)

        # Assuming the output is a single tensor representing embeddings
        # The key for the output can vary; check TRTEngine output names if needed.
        # For simplicity, assume it's the first output in raw.values()
        embedding_tensor = list(raw.values())[0]

        # BUGFIX: the raw w600k_r50 output is NOT unit length. Cosine similarity (and the THRESHOLD
        # comparison in the pipeline, which uses `embs @ db_gpu.T`) needs L2-normalised vectors.
        return F.normalize(embedding_tensor.float(), p=2, dim=1)

## 5) Build TensorRT engines and load the models

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Engines are built directly into (and reloaded from) Drive, so they survive a runtime restart.
ENGINE_DIR = "/content/drive/MyDrive/face_recognition_project/engines"
os.makedirs(ENGINE_DIR, exist_ok=True)
SCRFD_ENGINE = os.path.join(ENGINE_DIR, os.path.basename(SCRFD_ENGINE))
W600K_ENGINE = os.path.join(ENGINE_DIR, os.path.basename(W600K_ENGINE))

print("SCRFD engine:", SCRFD_ENGINE)
print("W600K engine:", W600K_ENGINE)


In [ ]:
# ============================================================
# SCRFD ONNX simplification + TensorRT engine build + model load
# SCRFD uses a fixed batch size of 1; W600K uses a dynamic batch.
# Engines that already exist (in Drive) are reused, not rebuilt.
# ============================================================
import onnx

# 1. Simplify the SCRFD ONNX (only needed when the engine has to be built)
scrfd_onnx_for_trt = SCRFD_ONNX
if not os.path.exists(SCRFD_ENGINE):
    simplified_scrfd_onnx = SCRFD_ONNX.replace(".onnx", "_simplified.onnx")
    if os.path.exists(simplified_scrfd_onnx):
        print("Simplified ONNX already exists:", simplified_scrfd_onnx)
        scrfd_onnx_for_trt = simplified_scrfd_onnx
    else:
        try:
            import onnxsim
            print(f"Simplifying ONNX model: {SCRFD_ONNX} -> {simplified_scrfd_onnx}")
            simplified_model, ok = onnxsim.simplify(onnx.load(SCRFD_ONNX))
            if ok:
                onnx.save(simplified_model, simplified_scrfd_onnx)
                scrfd_onnx_for_trt = simplified_scrfd_onnx
                print("SCRFD ONNX simplification successful.")
            else:
                print("ONNX simplification check failed; using the original SCRFD ONNX.")
        except Exception as e:
            print(f"ONNX simplification failed ({e}); using the original SCRFD ONNX.")

# 2. SCRFD TensorRT engine (fixed batch 1)
print("\nSCRFD TensorRT engine...")
build_engine(scrfd_onnx_for_trt, SCRFD_ENGINE,
             (1, 3, DET_SIZE, DET_SIZE), (1, 3, DET_SIZE, DET_SIZE), (1, 3, DET_SIZE, DET_SIZE),
             fp16=False)

# 3. W600K TensorRT engine (dynamic batch)
print("\nW600K TensorRT engine...")
build_engine(W600K_ONNX, W600K_ENGINE,
             (EMB_BATCH[0], 3, 112, 112), (EMB_BATCH[1], 3, 112, 112), (EMB_BATCH[2], 3, 112, 112),
             fp16=USE_FP16)

# 4. Load the models ONCE
scrfd = SCRFDModel(SCRFD_ENGINE)
w600k = W600KModel(W600K_ENGINE)

# 5. Warm-up + sanity checks (first TensorRT calls are slower)
scrfd.predict([np.zeros((480, 640, 3), np.uint8)] * MAX_STREAMS)
_emb = w600k.predict([np.zeros((112, 112, 3), np.uint8)])
print("\nEmbedding shape:", tuple(_emb.shape), "| L2 norm (must be ~1.0):", float(_emb.norm(dim=1)[0]))

print("Models ready on:", torch.cuda.get_device_name(0))
print("Detector: InsightFace SCRFD det_10g | Embedder: W600K / w600k_r50")


In [ ]:
# ---------- GPU / inference-backend diagnostics ----------
print("CUDA available       :", torch.cuda.is_available())
print("GPU name             :", torch.cuda.get_device_name(0))
print("CUDA version (torch) :", torch.version.cuda)
print("TensorRT version     :", trt.__version__)
print("FP16 engines requested:", USE_FP16)
print("SCRFD engine file    :", SCRFD_ENGINE, "| exists:", os.path.exists(SCRFD_ENGINE))
print("W600K engine file    :", W600K_ENGINE, "| exists:", os.path.exists(W600K_ENGINE))
print("Face detector        : InsightFace SCRFD det_10g -> TensorRT -> CUDA")
print("Face embedder        : W600K w600k_r50 -> TensorRT -> CUDA")
print("Video decoder        : FFmpeg NVDEC -> BGR frames")
print("Cosine threshold     :", THRESHOLD)

print()
print("GPU memory allocated :", round(torch.cuda.memory_allocated() / 1024**2, 1), "MB")
print("GPU memory reserved  :", round(torch.cuda.memory_reserved() / 1024**2, 1), "MB")

# NVDEC is required by this flow. Probe the actual FFmpeg build.
try:
    hw  = subprocess.run(["ffmpeg", "-hide_banner", "-hwaccels"], capture_output=True, text=True, timeout=10).stdout.lower()
    dec = subprocess.run(["ffmpeg", "-hide_banner", "-decoders"], capture_output=True, text=True, timeout=10).stdout.lower()
except Exception as e:
    raise RuntimeError(f"NVDEC diagnostic failed (is ffmpeg installed?): {e}")
has_cuda, has_cuvid = "cuda" in hw, "cuvid" in dec
print("NVDEC FFmpeg CUDA hwaccel:", "AVAILABLE" if has_cuda else "NOT AVAILABLE")
print("NVDEC FFmpeg cuvid decoders:", "AVAILABLE" if has_cuvid else "NOT AVAILABLE")
if USE_NVDEC and not has_cuda:
    raise RuntimeError("This notebook requires FFmpeg CUDA/NVDEC support, but it is not available.")


In [ ]:
!nvidia-smi


## 6) Face crop helper (unchanged from the working version)

In [ ]:
def crop_face(img, box, margin=0.15):
    """
    Crop a square region around a detected face.
    - Makes the box square (embedding model expects 1:1 aspect ratio).
    - Adds `margin` (15% by default) of context on each side.
    - Clamps to image borders so we never index out of range.
    """
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box[:4]

    side = max(x2 - x1, y2 - y1) * (1 + margin * 2)   # square side incl. margin
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2             # face center

    x1, x2 = int(max(0, cx - side / 2)), int(min(w, cx + side / 2))
    y1, y2 = int(max(0, cy - side / 2)), int(min(h, cy + side / 2))
    return img[y1:y2, x1:x2]

## 7) Persistent / incremental gallery database

Upload a **.zip** (or images). Person name = subfolder name (`alice/1.jpg`) or filename (`alice.jpg`, `alice_2.jpg` → **alice**).

**Re-run this cell whenever you want to add more reference images.** Nothing is ever deleted: new images are added to `GALLERY_DIR`, duplicates (same image content, even under another filename) are skipped, and only the *new* images are run through SCRFD + W600K. Embeddings are stored in `GALLERY_DB` and reused.

In [ ]:
import hashlib, shutil, tempfile
from google.colab import files

# --- Models must already be loaded by the section-5 build cell (no duplicate init here) ---
if "scrfd" not in globals() or "w600k" not in globals():
    raise RuntimeError("Models are not loaded - run the section 5 build cell first.")

# ======================================================================================
# PERSISTENT / INCREMENTAL GALLERY
#   GALLERY_DIR : every reference image ever uploaded (never cleared)
#   GALLERY_DB  : one row per image  -> (content hash, path, person, 512-d embedding)
#   The DB is keyed by the SHA-256 of the image *content*, so:
#     - the same image uploaded twice (even under another filename) is detected as a duplicate
#     - an image that was already embedded is never sent through SCRFD/W600K again
#     - only NEW images are detected + embedded; old embeddings are reused from the DB
#   The per-person mean matrix (`names`, `db_gpu`) used by the pipeline is rebuilt from the
#   stored embeddings after every update (cheap matrix math, no model inference).
# ======================================================================================
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
# If any of these change, old embeddings are no longer comparable -> DB is recomputed once.
GALLERY_SIGNATURE = f"{os.path.basename(W600K_ONNX)}|det{DET_SIZE}|score{MIN_DET_SCORE}|l2norm"


def file_sha256(path):
    """Content hash used for duplicate detection (independent of file name)."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def iter_images(root):
    """All image files below `root` (skips macOS zip junk such as __MACOSX/ and ._files)."""
    for p in sorted(glob.glob(os.path.join(root, "**", "*.*"), recursive=True)):
        if "__MACOSX" in p or os.path.basename(p).startswith("._"):
            continue
        if p.lower().endswith(IMG_EXTS):
            yield p


def person_of(path):
    """Same naming rule as before: sub-folder name, or file name (alice_2.jpg -> alice)."""
    parent = os.path.basename(os.path.dirname(path))
    if parent != os.path.basename(GALLERY_DIR):
        return parent
    return os.path.splitext(os.path.basename(path))[0].rsplit("_", 1)[0]


def scan_gallery():
    """{content_hash: path} of the images currently in GALLERY_DIR (first copy wins)."""
    on_disk = {}
    for p in iter_images(GALLERY_DIR):
        on_disk.setdefault(file_sha256(p), p)
    return on_disk


def load_gallery_db():
    """Load the persistent embedding DB -> {hash: {path, person, emb or None}}."""
    if not os.path.exists(GALLERY_DB):
        return {}
    try:
        z = np.load(GALLERY_DB, allow_pickle=False)
        if str(z["signature"]) != GALLERY_SIGNATURE:
            print("Gallery embedding settings changed -> recomputing all gallery embeddings once.")
            return {}
        return {str(h): dict(path=str(p), person=str(n), emb=(e if f else None))
                for h, p, n, e, f in zip(z["hashes"], z["paths"], z["persons"], z["embs"], z["has_face"])}
    except Exception as ex:
        print("Gallery DB unreadable (%s) -> it will be rebuilt from the gallery images." % ex)
        return {}


def save_gallery_db(records):
    """Write the DB atomically (temp file, then rename) so it can never be left half-written."""
    hs = list(records)
    embs = (np.stack([records[h]["emb"] if records[h]["emb"] is not None else np.zeros(512, np.float32)
                      for h in hs]).astype(np.float32) if hs else np.zeros((0, 512), np.float32))
    os.makedirs(os.path.dirname(GALLERY_DB), exist_ok=True)
    tmp = GALLERY_DB + ".tmp.npz"
    np.savez(tmp, signature=np.array(GALLERY_SIGNATURE), hashes=np.array(hs, dtype=str),
             paths=np.array([records[h]["path"] for h in hs], dtype=str),
             persons=np.array([records[h]["person"] for h in hs], dtype=str),
             embs=embs, has_face=np.array([records[h]["emb"] is not None for h in hs], dtype=bool))
    os.replace(tmp, GALLERY_DB)


def ingest_upload(uploaded_names):
    """
    Move newly uploaded files (images and/or .zip) into GALLERY_DIR WITHOUT touching what is
    already there. Images whose content is already in the gallery are skipped (duplicates).
    Returns (copied, duplicates).
    """
    known = set(scan_gallery())                 # content hashes already stored in the gallery
    tmp = tempfile.mkdtemp(prefix="gallery_upload_")
    candidates = []                             # (source file, path relative to gallery root)
    for name in uploaded_names:
        if name.lower().endswith(".zip"):
            root = tempfile.mkdtemp(dir=tmp)    # separate extraction folder per zip
            with zipfile.ZipFile(name) as z:
                z.extractall(root)
            candidates += [(p, os.path.relpath(p, root)) for p in iter_images(root)]
        elif name.lower().endswith(IMG_EXTS):
            candidates.append((name, os.path.basename(name)))
        else:
            print("ignored (not an image or .zip):", name)

    copied = dupes = 0
    for src, rel in candidates:
        h = file_sha256(src)
        if h in known:                          # same content already in gallery -> skip
            dupes += 1
            continue
        dst = os.path.join(GALLERY_DIR, rel)
        if os.path.exists(dst):                 # same name but DIFFERENT image -> keep both
            d, fn = os.path.split(rel)
            stem, ext = os.path.splitext(fn)
            # keep the person-part of the name so the person label does not change
            dst = os.path.join(GALLERY_DIR, d, f"{stem.rsplit('_', 1)[0]}_{h[:8]}{ext}")
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        known.add(h)
        copied += 1

    shutil.rmtree(tmp, ignore_errors=True)
    for name in uploaded_names:                 # remove the temporary upload copies in /content
        if os.path.isfile(name):
            os.remove(name)
    return copied, dupes


def sync_gallery(records):
    """
    Make the embedding DB consistent with the images on disk:
      - image on disk, not in DB  -> detect face + embed it (ONLY these go through the models)
      - image in DB, not on disk  -> drop its record (user deleted the file)
      - image in both             -> keep the stored embedding (no recomputation)
    Images with no usable face are stored with emb=None so they are not re-processed every time.
    Returns (added, no_face, removed).
    """
    on_disk = scan_gallery()
    removed = [h for h in records if h not in on_disk]
    for h in removed:
        del records[h]
    for h, p in on_disk.items():                # refresh path/person if a file was moved/renamed
        if h in records:
            records[h]["path"], records[h]["person"] = os.path.relpath(p, GALLERY_DIR), person_of(p)

    crops, crop_hashes, no_face = [], [], 0
    for h in (h for h in on_disk if h not in records):          # NEW images only
        p = on_disk[h]
        records[h] = dict(path=os.path.relpath(p, GALLERY_DIR), person=person_of(p), emb=None)
        img = cv2.imread(p)
        if img is None:
            print("unreadable image:", records[h]["path"]); no_face += 1; continue
        dets = scrfd.predict([img])[0]                            # SCRFD via TensorRT
        if not dets:
            print("no face found in:", records[h]["path"]); no_face += 1; continue
        best = max(dets, key=lambda d: (d[2] - d[0]) * (d[3] - d[1]))   # largest face
        crop = crop_face(img, best)
        if crop.size == 0:
            no_face += 1; continue
        crops.append(crop); crop_hashes.append(h)
    if crops:                                                    # ONE W600K call for all new faces
        for h, e in zip(crop_hashes, w600k.predict(crops).cpu().numpy()):
            records[h]["emb"] = e.astype(np.float32)
    return len(crop_hashes), no_face, len(removed)


def rebuild_gallery_matrix(records):
    """Per-person mean embedding (re-normalized) -> `names` + `db_gpu` used by the pipeline."""
    global names, db_gpu
    by_person = {}
    for r in records.values():
        if r["emb"] is not None:
            by_person.setdefault(r["person"], []).append(r["emb"])
    names = sorted(by_person)
    if names:
        db = np.stack([np.mean(by_person[n], axis=0) for n in names])
        db /= np.linalg.norm(db, axis=1, keepdims=True)
        db_gpu = torch.from_numpy(db).float().to(DEVICE)         # kept on GPU for fast cosine similarity
    else:
        db_gpu = torch.zeros((0, 512), device=DEVICE)


# ------------------------------- run: upload -> add -> update DB -------------------------------
os.makedirs(GALLERY_DIR, exist_ok=True)
gallery_records = load_gallery_db()             # embeddings from previous uploads (if any)

print("Upload NEW reference images / .zip (choose no file to just re-check the gallery).")
up = files.upload()
copied, dupes = ingest_upload(list(up))
added, no_face, removed = sync_gallery(gallery_records)
save_gallery_db(gallery_records)
rebuild_gallery_matrix(gallery_records)

total = sum(r["emb"] is not None for r in gallery_records.values())
print(f"\nAdded {added} new gallery images. {total} total gallery images available.")
if dupes:   print(f"  {dupes} duplicate image(s) skipped (already in the gallery).")
if no_face: print(f"  {no_face} new image(s) had no usable face and were not added to the embeddings.")
if removed: print(f"  {removed} image(s) no longer on disk were removed from the embedding DB.")
if not names: print("  WARNING: gallery is empty - every face will be labelled Unknown.")
print(f"Gallery ready: {len(names)} people -> {names}")


## 8) Lightweight IoU tracker (NEW — enables recognition caching)

No new dependency (plain Python/NumPy). This is deliberately simple: it exists to give the recognition-caching step below a stable `track_id` to key its cache on, not to replace a full tracker like ByteTrack. It matches new SCRFD detections to existing tracks by IoU, creates a new track for anything unmatched, and drops a track after `TRACK_MAX_MISSES` consecutive rounds without a match. Between detection rounds (`DETECTION_INTERVAL` > 1), `propagate()` reuses each track's last known box — there is no motion model, so boxes lag behind fast motion the larger `DETECTION_INTERVAL` is set.

In [ ]:
class SimpleTracker:
    """Minimal IoU-based tracker. Purpose: give recognition-caching (Step 4) a `track_id` to key on.
    Each track holds its own recognition cache: {identity, score, last_recognized_frame}."""

    def __init__(self, iou_thr=TRACK_IOU_THRESHOLD, max_misses=TRACK_MAX_MISSES):
        self.iou_thr, self.max_misses = iou_thr, max_misses
        self.tracks = {}          # track_id -> dict(bbox, misses, identity, score, last_recognized_frame)
        self._next_id = 0

    @staticmethod
    def _iou(a, b):
        ax1, ay1, ax2, ay2 = a[:4]; bx1, by1, bx2, by2 = b[:4]
        ix1, iy1 = max(ax1, bx1), max(ay1, by1)
        ix2, iy2 = min(ax2, bx2), min(ay2, by2)
        iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
        inter = iw * ih
        area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
        area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
        union = area_a + area_b - inter
        return inter / union if union > 0 else 0.0

    def update_with_detections(self, dets):
        """New SCRFD detections this round. Returns [(track_id, bbox, is_new_track), ...]."""
        pairs = []
        for tid, t in self.tracks.items():
            for di, d in enumerate(dets):
                iou = self._iou(t["bbox"], d)
                if iou >= self.iou_thr:
                    pairs.append((iou, tid, di))
        pairs.sort(key=lambda p: p[0], reverse=True)          # greedy best-IoU-first matching

        matched_tids, used_dets, assignments = set(), set(), []
        for iou, tid, di in pairs:
            if tid in matched_tids or di in used_dets:
                continue
            matched_tids.add(tid); used_dets.add(di)
            self.tracks[tid]["bbox"], self.tracks[tid]["misses"] = dets[di], 0
            assignments.append((tid, dets[di], False))

        for di, d in enumerate(dets):                          # unmatched detections -> new tracks
            if di in used_dets:
                continue
            tid, self._next_id = self._next_id, self._next_id + 1
            self.tracks[tid] = dict(bbox=d, misses=0, identity=None, score=0.0,
                                     last_recognized_frame=-1)
            assignments.append((tid, d, True))

        for tid in self.tracks:                                 # age unmatched existing tracks
            if tid not in matched_tids and not any(a[0] == tid for a in assignments):
                self.tracks[tid]["misses"] += 1
        # NOTE: dead tracks are NOT deleted here -- see prune(). A chunk buffers assignments for
        # many frames before recognition/drawing reads them back by track_id; deleting mid-chunk
        # caused a KeyError if a track went stale partway through a chunk it was still needed in.
        return assignments

    def propagate(self):
        """No detection this round: reuse each live track's last known box unchanged."""
        return [(tid, t["bbox"], False) for tid, t in self.tracks.items()
                if t["misses"] <= self.max_misses]

    def prune(self):
        """Actually drop tracks that exceeded max_misses. Call this ONCE per processing chunk,
        after every pass that might still reference a track_id assigned earlier in that same
        chunk (recognition, drawing) -- never mid-chunk."""
        self.tracks = {tid: t for tid, t in self.tracks.items() if t["misses"] <= self.max_misses}


## 9) FPS-sampling helpers + single-video processing function (chunk-batched for real-time speed)

`valid_fps` / `frame_slot` / `keep_frame` / `sampling_plan` are unchanged -- they decide which frames are decoded at all when input FPS exceeds `TARGET_FPS`.

`process_single_video` now buffers `FRAME_CHUNK_SIZE` frames at a time. Within each chunk:
1. Every detection-due frame (`DETECTION_INTERVAL`) is batched into **one** SCRFD TensorRT call instead of one call per frame.
2. Every recognition-due crop across the whole chunk is batched into **one** W600K TensorRT call instead of one call per frame.
3. The IoU tracker still processes frames strictly in temporal order within the chunk, so track continuity is unaffected by the batching -- only the *inference calls* are batched, not the tracking logic.

This matters because TensorRT pays a fixed per-call overhead (kernel launch, shape/context setup) close to independent of batch size. Calling `.predict()` once per frame -- even with `DETECTION_INTERVAL`/`RECOGNITION_INTERVAL` already skipping most frames -- still paid that overhead on every frame that *did* need inference. Batching many of those frames into one call is the main lever left for reaching real-time throughput; it changes nothing about detection/recognition accuracy or the output video timeline, which are both identical to the unbatched version.

In [ ]:
# ============================================================
# VIDEO READER
# GPU decoding when available, OpenCV fallback for reliability
# ============================================================

import math
import subprocess
import numpy as np
import cv2


# ------------------------------------------------------------------
# FPS / frame sampling
# ------------------------------------------------------------------

def valid_fps(fps, default=25.0):
    """Container FPS can be 0/NaN -> use a safe default."""
    return float(fps) if fps and math.isfinite(fps) and fps > 0 else default


def frame_slot(idx, ratio):
    """Index of the TARGET_FPS tick containing source frame idx."""
    return math.floor(idx * ratio + 1e-9)


def keep_frame(idx, ratio):
    """Keep a frame only when it starts a new sampling tick."""
    return frame_slot(idx, ratio) > frame_slot(idx - 1, ratio)


def sampling_plan(fps, total_frames):
    """Create frame-sampling plan using the video's real FPS."""
    ratio = min(1.0, TARGET_FPS / fps)
    effective_fps = fps * ratio

    to_process = (
        frame_slot(total_frames - 1, ratio) + 1
        if total_frames > 0 else 0
    )

    return dict(
        ratio=ratio,
        effective_fps=effective_fps,
        to_process=to_process,
        to_skip=max(0, total_frames - to_process)
    )


# ============================================================
# NVDEC VIDEO READER
# ============================================================

class NVDECVideoReader:
    """
    Video reader used by the pipeline.

    Tries FFmpeg with CUDA/NVDEC hardware decoding first. If FFmpeg cannot produce even ONE frame
    (no NVDEC, unsupported codec, ...) it falls back to OpenCV decoding so the pipeline keeps working.
    `self.backend` is "nvdec" or "opencv" so you can see which decoder was really used.
    """

    def __init__(self, path, width, height):
        self.path = path
        self.width = int(width)
        self.height = int(height)
        self.frame_bytes = self.width * self.height * 3

        self.proc = None
        self.cap = None
        self.frames_from_ffmpeg = 0
        self.backend = "opencv"

        # NVDEC decodes on the GPU; frames are copied back to system memory and converted to BGR24.
        # (No `-hwaccel_output_format cuda` + `hwdownload,format=bgr24`: hwdownload can only output the
        #  decoder's native format, e.g. nv12, so forcing bgr24 there makes FFmpeg fail.)
        cmd = [
            "ffmpeg", "-hide_banner", "-loglevel", "error",
            "-hwaccel", "cuda",
            "-i", path,
            "-an",
            "-f", "rawvideo", "-pix_fmt", "bgr24",
            "pipe:1",
        ]
        try:
            # stderr is discarded: an unread stderr PIPE can fill up and deadlock FFmpeg.
            self.proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, bufsize=10**8)
            self.backend = "nvdec"
        except Exception:
            self.proc = None

        if self.proc is None:
            self._open_opencv()

    def _open_opencv(self):
        self.cap = cv2.VideoCapture(self.path)
        if not self.cap.isOpened():
            self.cap = None
            raise RuntimeError(f"Could not open video: {self.path}")
        self.backend = "opencv"

    def read(self):
        """Returns (True, BGR frame) or (False, None) at the real end of the video."""
        if self.proc is not None:
            data = self.proc.stdout.read(self.frame_bytes)
            if len(data) == self.frame_bytes:
                self.frames_from_ffmpeg += 1
                frame = np.frombuffer(data, dtype=np.uint8).reshape(self.height, self.width, 3).copy()
                return True, frame

            # Short read: either NVDEC failed, or the stream simply ended.
            self._close_ffmpeg()
            if self.frames_from_ffmpeg > 0:
                return False, None            # BUGFIX: genuine end of stream -> do NOT restart with OpenCV
            try:                              # NVDEC produced nothing -> decode the whole file with OpenCV
                self._open_opencv()
            except RuntimeError:
                return False, None

        if self.cap is not None:
            ok, frame = self.cap.read()
            if not ok or frame is None:
                return False, None
            return True, frame

        return False, None

    def _close_ffmpeg(self):
        if self.proc is None:
            return
        try:
            if self.proc.poll() is None:
                self.proc.terminate()
                try:
                    self.proc.wait(timeout=2)
                except subprocess.TimeoutExpired:
                    self.proc.kill()
            if self.proc.stdout:
                self.proc.stdout.close()
        except Exception:
            pass
        self.proc = None

    def close(self):
        self._close_ffmpeg()
        if self.cap is not None:
            try:
                self.cap.release()
            except Exception:
                pass
            self.cap = None


In [ ]:
def process_single_video(path, out_path,
                          detection_interval=DETECTION_INTERVAL,
                          recognition_interval=RECOGNITION_INTERVAL,
                          min_recog_score=MIN_RECOG_SCORE,
                          chunk_size=FRAME_CHUNK_SIZE):
    """Process ONE video end-to-end, buffering `chunk_size` frames at a time so every
    detection-due and recognition-due frame within a chunk is sent to TensorRT as ONE batched
    call instead of one call per frame (see FRAME_CHUNK_SIZE comment in the config cell -- this
    is the main change for real-time speed). Tracking still runs strictly in temporal order
    within each chunk, so track IDs and box continuity are unaffected by the batching.

    ASYNC DECODE/GPU OVERLAP (NEW): chunk N+1 is decoded on a background CPU thread (cv2 releases
    the GIL during I/O-bound read()/grab() calls) WHILE chunk N is going through detection /
    tracking / recognition / drawing / writing on the main thread. Only the background thread ever
    touches `cap` and only the main thread ever touches the GPU models / tracker / writer, so this
    adds overlap without any shared-mutable-state races. A bounded queue (maxsize=2) caps how many
    decoded chunks can sit in host RAM ahead of the GPU, per Step 8's "no unbounded buffering" rule.
    Frame order is unaffected: the queue is strictly FIFO, so output frame order is identical to the
    fully-serial version -- this only removes idle time, it changes nothing about what is computed.
    Falls back to the old fully-serial loop if `USE_ASYNC_PIPELINE` (config cell) is False.
    """
    require_nvidia_gpu()
    timers = dict(read=0.0, detect=0.0, track=0.0, recognize=0.0, write=0.0, gpu_wait_for_decode=0.0)
    counts = dict(frames_read=0, frames_processed=0, detection_rounds=0, detection_batches=0,
                  recognition_ops=0, recognition_batches=0)

    meta_cap = cv2.VideoCapture(path)
    if not meta_cap.isOpened():
        raise IOError(f"cannot open {path!r} for metadata")
    fps    = valid_fps(meta_cap.get(cv2.CAP_PROP_FPS))
    width  = int(meta_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(meta_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(meta_cap.get(cv2.CAP_PROP_FRAME_COUNT))
    meta_cap.release()
    reader = NVDECVideoReader(path, width, height)
    plan   = sampling_plan(fps, total)
    out_fps = plan["effective_fps"]

    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), out_fps, (width, height))
    if not writer.isOpened():
        reader.close(); raise IOError(f"cannot create writer for {out_path!r}")

    tracker = SimpleTracker()
    idx = processed = 0
    t0 = time.perf_counter()

    def read_chunk():
        """Read up to `chunk_size` KEPT frames from NVDEC.

        NVDEC performs the decode. Source frames that are not selected by TARGET_FPS are decoded
        and discarded so the existing sampling/output timeline remains unchanged.
        """
        nonlocal idx
        buf = []
        while len(buf) < chunk_size:
            t_r = time.perf_counter()
            ok, frame = reader.read()
            timers["read"] += time.perf_counter() - t_r
            if not ok:
                break
            counts["frames_read"] += 1
            source_idx = idx
            idx += 1
            # BUGFIX: apply the TARGET_FPS sampling rule (same as the concurrent path). Before, every
            # frame was kept, so >30 FPS sources were written at a lower FPS (slow-motion output).
            if frame is not None and keep_frame(source_idx, plan["ratio"]):
                buf.append(frame)
        return buf

    def process_chunk(frames):
        """Everything that happens to one already-decoded chunk of frames: batched detection,
        temporal tracking, batched recognition, draw + write. Unchanged logic from the fully-serial
        version -- only WHEN this runs relative to decoding of the next chunk has changed."""
        nonlocal processed

        # ---- batched detection: ONE TensorRT call for every detection-due frame in chunk ----
        det_positions = [i for i in range(len(frames)) if (processed + i) % detection_interval == 0]
        t_d = time.perf_counter()
        dets_by_pos = {}
        if det_positions:
            batch_dets = scrfd.predict([frames[i] for i in det_positions])
            dets_by_pos = dict(zip(det_positions, batch_dets))
            counts["detection_rounds"] += len(det_positions)
            counts["detection_batches"] += 1
        timers["detect"] += time.perf_counter() - t_d

        # ---- tracker still runs strictly in temporal order (correctness depends on this) ----
        t_t = time.perf_counter()
        per_frame_assigns = []
        for i in range(len(frames)):
            if i in dets_by_pos:
                assigns = tracker.update_with_detections(list(dets_by_pos[i]))
            else:
                assigns = tracker.propagate()
            per_frame_assigns.append(assigns)
        timers["track"] += time.perf_counter() - t_t

        # ---- collect every recognition-due crop across the WHOLE chunk, ONE TensorRT call ----
        t_rec = time.perf_counter()
        to_recognize = []   # (frame_pos_in_chunk, track_id, crop)
        for i, assigns in enumerate(per_frame_assigns):
            frame_processed_idx = processed + i
            for tid, bbox, is_new in assigns:
                trk = tracker.tracks[tid]
                stale    = (frame_processed_idx - trk["last_recognized_frame"]) >= recognition_interval
                low_conf = trk["identity"] is None or trk["score"] < min_recog_score
                if is_new or low_conf or stale:
                    c = crop_face(frames[i], bbox)
                    if c.size == 0:
                        continue
                    to_recognize.append((i, tid, c))
                    trk["last_recognized_frame"] = frame_processed_idx  # avoid re-queuing same
                                                                         # track later in this chunk
        if to_recognize:
            crops = [c for _, _, c in to_recognize]
            if len(names):
                embs = w600k.predict(crops)
                sims = embs @ db_gpu.T
                best_sc, best_j = sims.max(dim=1)
                best_sc, best_j = best_sc.cpu().numpy(), best_j.cpu().numpy()
                for (_, tid, _), sc, j in zip(to_recognize, best_sc, best_j):
                    trk = tracker.tracks[tid]
                    trk["identity"], trk["score"] = (names[j] if sc >= THRESHOLD else "Unknown"), float(sc)
            else:
                for _, tid, _ in to_recognize:
                    tracker.tracks[tid]["identity"], tracker.tracks[tid]["score"] = "Unknown", 0.0
            counts["recognition_ops"] += len(to_recognize)
            counts["recognition_batches"] += 1
        timers["recognize"] += time.perf_counter() - t_rec

        # ---- draw + write, one sequential pass so output frame order is exactly preserved ----
        for i, assigns in enumerate(per_frame_assigns):
            frame = frames[i]
            for tid, bbox, is_new in assigns:
                trk = tracker.tracks[tid]
                x1, y1, x2, y2 = map(int, bbox[:4])
                label, sc = (trk["identity"] or "Unknown"), trk["score"]
                color = (0, 200, 0) if label != "Unknown" else (0, 0, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f"{label} {sc*100:.0f}%", (x1, max(15, y1 - 8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            t_w = time.perf_counter()
            writer.write(frame)
            timers["write"] += time.perf_counter() - t_w
            counts["frames_processed"] += 1
        tracker.prune()   # safe now: every pass that needed this chunk's track_ids is done
        processed += len(frames)

    use_async = globals().get("USE_ASYNC_PIPELINE", True)
    stop = threading.Event()
    prod_err = []
    prod_thread = None

    try:
        if use_async:
            # ---- producer/consumer overlap: decode chunk N+1 while chunk N is on the GPU ----
            chunk_q = queue.Queue(maxsize=2)          # bounded -> at most 2 chunks of host RAM ahead

            def _put(item):
                while not stop.is_set():
                    try:
                        chunk_q.put(item, timeout=0.1)
                        return True
                    except queue.Full:
                        pass
                return False

            def producer():
                try:
                    while True:
                        buf = read_chunk()
                        if not _put(buf) or not buf:
                            return
                except Exception as e:                # never leave the consumer waiting forever
                    prod_err.append(e)
                    _put([])

            prod_thread = threading.Thread(target=producer, daemon=True)
            prod_thread.start()
            while True:
                t_wait = time.perf_counter()
                frames = chunk_q.get()
                timers["gpu_wait_for_decode"] += time.perf_counter() - t_wait
                if not frames:
                    break
                process_chunk(frames)
            prod_thread.join()
            if prod_err:
                raise prod_err[0]
        else:
            # ---- fully-serial fallback ----
            while True:
                frames = read_chunk()
                if not frames:
                    break
                process_chunk(frames)
    finally:
        stop.set()
        reader.close()
        if prod_thread is not None:
            prod_thread.join(timeout=5)
        writer.release()

    wall = time.perf_counter() - t0
    gpu_mem_mb = torch.cuda.memory_allocated() / 1024**2       # one-time read, not per-frame
    stats = dict(video=os.path.basename(path), width=width, height=height, input_fps=fps,
                 effective_fps=out_fps, duration=counts["frames_read"] / fps if fps > 0 else 0.0,
                 wall=wall, gpu_mem_mb=gpu_mem_mb, decoder=reader.backend, **timers, **counts,
                 processing_fps=counts["frames_processed"] / wall if wall > 0 else 0.0)
    stats["speed_ratio"] = stats["processing_fps"] / out_fps if out_fps > 0 else 0.0
    stats["real_time"]   = stats["processing_fps"] >= out_fps
    return stats


## 10) Batch queue — upload and process 1 to 30 videos (Step 12/13)

Videos are processed **one at a time**, reusing the already-loaded `scrfd` / `w600k` models and the same `process_single_video()` function above — no simultaneous decoding of multiple videos, so GPU/RAM usage stays the same as processing a single video. A failure on one video is caught, reported, and does not stop the rest of the queue.

In [ ]:
from google.colab import files as gfiles

print(f"Select 1 to {MAX_VIDEOS_PER_BATCH} video files in the picker (multi-select is supported --")
print("you do not need to wait for one video to finish before choosing the next; all files you")
print("select are registered together in this one upload action -- Step 4).")
up = gfiles.upload()
VIDEO_PATHS = sorted(up.keys())
if not (1 <= len(VIDEO_PATHS) <= MAX_VIDEOS_PER_BATCH):
    raise ValueError(f"Please provide between 1 and {MAX_VIDEOS_PER_BATCH} videos (got {len(VIDEO_PATHS)}).")

print(f"\nTotal Videos: {len(VIDEO_PATHS)}\n")
for i, p in enumerate(VIDEO_PATHS, 1):
    size_mb = len(up[p]) / (1024 ** 2)
    print(f"  Video {i:02d}  {p:<40s}  Uploaded  ({size_mb:.1f} MB)")


In [ ]:
# ---------- GPU video encoding, auto-detected (Step 13 / Steps 4-6 of the parallelism request) ----------
# Only switches to h264_nvenc if THIS Colab's ffmpeg build actually has it -- verified at runtime,
# never assumed. Falls back to libx264 (CPU, -preset ultrafast) if NVENC isn't available or fails.
def detect_video_encoder():
    try:
        out = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"],
                              capture_output=True, text=True, check=True).stdout
        if "h264_nvenc" in out:
            return ["-vcodec", "h264_nvenc", "-preset", "p1", "-pix_fmt", "yuv420p"]
    except Exception as e:
        print(f"  NVENC probe failed ({e}); using CPU encoder.")
    return ["-vcodec", "libx264", "-preset", "ultrafast", "-pix_fmt", "yuv420p"]


VIDEO_ENCODER_ARGS = detect_video_encoder()
print(f"Video encoder: {VIDEO_ENCODER_ARGS[1]}"
      f"{' (GPU/NVENC)' if VIDEO_ENCODER_ARGS[1] == 'h264_nvenc' else ' (CPU)'}")


def encode_to_h264(r):
    """Re-encode ONE completed video's raw mp4v output to H.264 (mp4v doesn't play in-browser).
    Runs as an external ffmpeg process, so it does not hold the Python GIL or contend with TensorRT's
    CUDA compute -- this is what lets it run in the background while the NEXT video is already being
    decoded/detected/recognized on the GPU (Steps 4/6: real overlap, not N independent GPU workers)."""
    out_final = f"annotated_{os.path.splitext(r['video'])[0]}.mp4"
    t_enc = time.perf_counter()
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", r["out_raw"], *VIDEO_ENCODER_ARGS, out_final]
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as e:
        if VIDEO_ENCODER_ARGS[1] == "h264_nvenc":
            print(f"  NVENC encode failed for {r['video']}, retrying with CPU libx264: {e}")
            cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", r["out_raw"],
                   "-vcodec", "libx264", "-preset", "ultrafast", "-pix_fmt", "yuv420p", out_final]
            try:
                subprocess.run(cmd, check=True)
            except Exception as e2:
                r["status"], r["error"] = "Failed", f"encode failed (both NVENC and CPU): {e2}"
                return r
        else:
            r["status"], r["error"] = "Failed", f"encode failed: {e}"
            return r
    r["encode_time"] = time.perf_counter() - t_enc
    r["total_time"]  = r["wall"] + r["encode_time"]
    r["out_final"]   = out_final
    return r


def print_queue_status(statuses, paths):
    """Pending / Processing / Completed / Failed table (Step 5)."""
    print("  " + " | ".join(f"{os.path.basename(p)}: {statuses[p]}" for p in paths))


def process_video_batch(paths, detection_interval=DETECTION_INTERVAL,
                         recognition_interval=RECOGNITION_INTERVAL):
    """GPU processing stays a SINGLE controlled sequential queue over `paths` -- exactly one video's
    frames are ever on the TensorRT engines / tracker at a time (Step 6: never N parallel GPU workers,
    that contends for VRAM/CUDA-context and can make things slower). The overlap this notebook adds
    is at the OTHER end of the pipeline: once a video's GPU work is done, its CPU/NVENC H.264 encode
    is handed to a small background thread pool (MAX_CONCURRENT_ENCODES workers) and GPU processing
    immediately moves on to the next queued video instead of waiting for that encode to finish.
    Frame order/timeline and tracking identity inside each video are completely unaffected -- this
    only changes when the encode of an ALREADY-FINISHED video's frames happens relative to the next
    video's GPU work. Returns a list of per-video stats dicts (status="Completed"/"Failed")."""
    n = len(paths)
    results = []
    statuses = {p: "Pending" for p in paths}
    encode_pool = ThreadPoolExecutor(max_workers=MAX_CONCURRENT_ENCODES)
    encode_futures = []

    try:
        for i, p in enumerate(paths):
            statuses[p] = "Processing"
            print(f"\n[{i+1}/{n}] Processing {os.path.basename(p)} ...")
            print_queue_status(statuses, paths)
            out_raw = f"annotated_{os.path.splitext(os.path.basename(p))[0]}_raw.mp4"
            try:
                stats = process_single_video(p, out_raw, detection_interval, recognition_interval)
                stats["status"], stats["out_raw"], stats["error"] = "Completed", out_raw, None
                print(f"  GPU stage done in {stats['wall']:.1f}s | "
                      f"{stats['processing_fps']:.1f} proc FPS | "
                      f"{stats['speed_ratio']:.2f}x realtime -> encoding in background")
                # Hand the CPU/NVENC encode to the background pool; GPU moves straight to next video.
                encode_futures.append(encode_pool.submit(encode_to_h264, stats))
                statuses[p] = "Completed (encoding)"
            except Exception as e:
                stats = dict(video=os.path.basename(p), status="Failed", error=str(e), out_raw=None)
                statuses[p] = "Failed"
                print(f"  FAILED: {e}")
            results.append(stats)

        # Drain background encodes (each future returns the stats dict, now carrying encode_time/out_final)
        by_video = {r["video"]: r for r in results}
        for fut in encode_futures:
            r = fut.result()
            by_video[r["video"]].update(r)
        for p in paths:
            r = by_video.get(os.path.basename(p))
            if r is not None:
                statuses[p] = r["status"]
    finally:
        encode_pool.shutdown(wait=True)

    completed = sum(r["status"] == "Completed" for r in results)
    failed    = sum(r["status"] == "Failed" for r in results)
    print(f"\nQueue finished -> Total: {n} | Completed: {completed} | Failed: {failed}")
    return results


# =====================================================================================
# CONCURRENT MULTI-STREAM PROCESSING  (N videos at once -> ONE shared GPU worker)
# =====================================================================================
#   Video 1..N -> [reader thread + bounded frame buffer each] -> ONE GPU worker loop
#   GPU worker each round: take the newest group of frames from every active stream,
#   run SCRFD ONCE on the latest frame of each stream (a single cross-stream TensorRT batch),
#   track in order, run W600K ONCE for every recognition-due crop of every stream,
#   then draw/write each stream's frames IN ORDER to its own output (N concurrent outputs).
#
#   "Skip old frames, process the latest": between detections a stream's frames are NOT sent to
#   TensorRT -- they reuse the tracker state (propagate) -- so e.g. frame 3 = detect, 4..7 = reuse,
#   8 = latest -> detect. For FILES every frame is still WRITTEN to the output (timeline/FPS/duration
#   preserved); only the expensive inference is skipped. For true LIVE feeds set LIVE_MODE=True: the
#   reader then drops the OLDEST buffered frame when the GPU falls behind (so output stays "now").
#   Only ONE thread (the GPU worker) ever touches TensorRT / trackers / writers -> no CUDA contention.
from types import SimpleNamespace
from collections import deque


def _put_frame(s, item):
    """Bounded put. File mode: block (back-pressure, never drops). Live mode: drop OLDEST if full."""
    while not s.cancel.is_set():
        try:
            s.q.put(item, timeout=0.05) if not s.live else s.q.put_nowait(item)
            return
        except queue.Full:
            if s.live and item is not None:
                try: s.q.get_nowait(); s.dropped += 1
                except queue.Empty: pass


def _stream_reader(s):
    """Owns the stream's FFmpeg/NVDEC reader exclusively."""
    try:
        while not s.cancel.is_set():
            t = time.perf_counter()
            ok, frame = s.reader.read()
            s.t_read += time.perf_counter() - t
            if not ok:
                break
            source_idx = s.idx
            s.frames_read += 1; s.idx += 1
            # Keep exactly the same TARGET_FPS sampling rule as the single-video path.
            if frame is not None and keep_frame(source_idx, s.plan["ratio"]):
                _put_frame(s, frame)
    except Exception as e:
        s.error = e
    finally:
        # sentinel must never be dropped, even in live mode
        while not s.cancel.is_set():
            try: s.q.put(None, timeout=0.05); break
            except queue.Full:
                if s.live:
                    try: s.q.get_nowait(); s.dropped += 1
                    except queue.Empty: pass


def _open_stream(path, out_raw):
    meta_cap = cv2.VideoCapture(path)  # metadata only; pixels are decoded by NVDEC below
    if not meta_cap.isOpened():
        raise IOError(f"cannot open {path!r} for metadata")
    fps    = valid_fps(meta_cap.get(cv2.CAP_PROP_FPS))
    width  = int(meta_cap.get(cv2.CAP_PROP_FRAME_WIDTH)); height = int(meta_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(meta_cap.get(cv2.CAP_PROP_FRAME_COUNT))
    meta_cap.release()
    plan   = sampling_plan(fps, total)
    reader = NVDECVideoReader(path, width, height)
    writer = cv2.VideoWriter(out_raw, cv2.VideoWriter_fourcc(*"mp4v"), plan["effective_fps"], (width, height))
    if not writer.isOpened():
        reader.close(); raise IOError(f"cannot create writer for {out_raw!r}")
    s = SimpleNamespace(
        path=path, name=os.path.basename(path), out_raw=out_raw, reader=reader, writer=writer,
        fps=fps, width=width, height=height, plan=plan, tracker=SimpleTracker(),
        q=queue.Queue(maxsize=STREAM_QUEUE_DEPTH), live=LIVE_MODE, cancel=threading.Event(),
        idx=0, frames_read=0, processed=0, dropped=0, done=False, error=None,
        never_detected=True, since_det=0, t0=time.perf_counter(), t_read=0.0,
        t_detect=0.0, t_track=0.0, t_recog=0.0, t_write=0.0,
        n_det=0, n_det_batches=0, n_rec=0, n_rec_batches=0, n_alerts=0)
    s.thread = threading.Thread(target=_stream_reader, args=(s,), daemon=True)
    s.thread.start()
    return s


def _close_stream(s):
    s.cancel.set()
    s.thread.join(timeout=2)
    s.reader.close(); s.writer.release()


def _stream_stats(s):
    wall = time.perf_counter() - s.t0
    out_fps = s.plan["effective_fps"]
    st = dict(video=s.name, width=s.width, height=s.height, input_fps=s.fps, effective_fps=out_fps,
              duration=s.frames_read / s.fps if s.fps > 0 else 0.0, wall=wall,
              gpu_mem_mb=torch.cuda.memory_allocated() / 1024**2,
              read=s.t_read, detect=s.t_detect, track=s.t_track, recognize=s.t_recog, write=s.t_write,
              gpu_wait_for_decode=0.0, frames_read=s.frames_read, frames_processed=s.processed,
              detection_rounds=s.n_det, detection_batches=s.n_det_batches,
              recognition_ops=s.n_rec, recognition_batches=s.n_rec_batches,
              decoder=s.reader.backend, frames_dropped_live=s.dropped, unknown_alerts=s.n_alerts,
              processing_fps=s.processed / wall if wall > 0 else 0.0)
    st["speed_ratio"] = st["processing_fps"] / out_fps if out_fps > 0 else 0.0
    st["real_time"] = st["processing_fps"] >= out_fps
    return st


def process_videos_concurrent(paths, detection_interval=DETECTION_INTERVAL,
                               recognition_interval=RECOGNITION_INTERVAL,
                               min_recog_score=MIN_RECOG_SCORE, max_streams=CONCURRENT_STREAMS):
    """Process up to `max_streams` videos AT THE SAME TIME through one shared GPU worker; when one
    finishes, the next queued video takes its slot (so 30 uploads never means 30 open streams).
    Returns per-video stats dicts (same shape as process_video_batch) plus a shared `alerts` list."""
    require_nvidia_gpu()
    pending, active, results, alerts = deque(paths), [], [], []
    statuses = {p: "Queued" for p in paths}
    encode_pool, encode_futures = ThreadPoolExecutor(max_workers=MAX_CONCURRENT_ENCODES), []
    by_name = {}
    round_frames = max(1, detection_interval)
    t_all = time.perf_counter()

    def fail(s, msg):
        _close_stream(s)
        statuses[s.path] = "Failed"
        r = dict(video=s.name, status="Failed", error=msg, out_raw=None)
        results.append(r); by_name[s.name] = r
        print(f"  FAILED {s.name}: {msg}")
        if s in active: active.remove(s)

    def finish(s):
        _close_stream(s)
        if s.error is not None:
            fail_stats = dict(video=s.name, status="Failed", error=f"decode error: {s.error}", out_raw=None)
            statuses[s.path] = "Failed"; results.append(fail_stats); by_name[s.name] = fail_stats
            print(f"  FAILED {s.name}: decode error: {s.error}")
        else:
            st = _stream_stats(s)
            st["status"], st["out_raw"], st["error"] = "Completed", s.out_raw, None
            results.append(st); by_name[s.name] = st
            statuses[s.path] = "Completed (encoding)"
            print(f"  Completed {s.name}: {st['wall']:.1f}s | {st['speed_ratio']:.2f}x realtime | "
                  f"{s.processed} frames written | {s.n_alerts} unknown alert(s) -> encoding in background")
            encode_futures.append(encode_pool.submit(encode_to_h264, st))
        active.remove(s)

    try:
        while pending or active:
            # ---- fill free stream slots from the queue ----
            while pending and len(active) < max_streams:
                p = pending.popleft()
                try:
                    s = _open_stream(p, f"annotated_{os.path.splitext(os.path.basename(p))[0]}_raw.mp4")
                    active.append(s); statuses[p] = "Processing"
                except Exception as e:
                    statuses[p] = "Failed"
                    r = dict(video=os.path.basename(p), status="Failed", error=str(e), out_raw=None)
                    results.append(r); by_name[r["video"]] = r
                    print(f"  FAILED to open {p}: {e}")
            if not active:
                continue

            # ---- 1. gather the NEWEST group of frames from every active stream (non-blocking) ----
            groups = []      # (stream, frames, det_pos or None)
            for s in active:
                frames = []
                while len(frames) < round_frames and not s.done:
                    try: item = s.q.get_nowait()
                    except queue.Empty: break
                    if item is None: s.done = True; break
                    frames.append(item)
                if frames:
                    # first group: detect on frame 0 so boxes show immediately; afterwards detect on the
                    # LATEST frame of the group once the interval has elapsed, else just reuse tracks
                    if s.never_detected:            det_pos = 0
                    elif s.since_det + len(frames) >= detection_interval: det_pos = len(frames) - 1
                    else:                           det_pos = None
                    groups.append((s, frames, det_pos))
            if not groups:
                for s in [x for x in active if x.done]: finish(s)
                time.sleep(0.002)
                continue

            try:
                # ---- 2. ONE cross-stream SCRFD batch: the latest frame of each stream that is due ----
                t = time.perf_counter()
                det_jobs = [(s, frames[dp]) for s, frames, dp in groups if dp is not None]
                det_result = {}
                if det_jobs:
                    outs = scrfd.predict([f for _, f in det_jobs])
                    for (s, _), d in zip(det_jobs, outs): det_result[id(s)] = list(d)
                dt = time.perf_counter() - t
                for s, _ in det_jobs:
                    s.t_detect += dt / len(det_jobs); s.n_det += 1; s.n_det_batches += 1

                # ---- 3. tracking, strictly in temporal order per stream ----
                t = time.perf_counter()
                assigns_by_stream = {}
                for s, frames, dp in groups:
                    per = []
                    for i in range(len(frames)):
                        per.append(s.tracker.update_with_detections(det_result[id(s)])
                                   if (dp is not None and i == dp) else s.tracker.propagate())
                    assigns_by_stream[id(s)] = per
                    if dp is not None:
                        s.never_detected = False; s.since_det = len(frames) - 1 - dp
                    else:
                        s.since_det += len(frames)
                dt = time.perf_counter() - t
                for s, _, _ in groups: s.t_track += dt / len(groups)

                # ---- 4. ONE cross-stream W600K batch for every recognition-due crop (cache-respecting) ----
                t = time.perf_counter()
                jobs = []   # (stream, tid, crop)
                for s, frames, _ in groups:
                    for i, assigns in enumerate(assigns_by_stream[id(s)]):
                        fidx = s.processed + i
                        for tid, bbox, is_new in assigns:
                            trk = s.tracker.tracks[tid]
                            stale = (fidx - trk["last_recognized_frame"]) >= recognition_interval
                            low = trk["identity"] is None or trk["score"] < min_recog_score
                            if is_new or low or stale:
                                c = crop_face(frames[i], bbox)
                                if c.size == 0: continue
                                jobs.append((s, tid, c)); trk["last_recognized_frame"] = fidx
                if jobs:
                    if len(names):
                        embs = w600k.predict([c for _, _, c in jobs])
                        best_sc, best_j = (embs @ db_gpu.T).max(dim=1)
                        best_sc, best_j = best_sc.cpu().numpy(), best_j.cpu().numpy()   # one D2H for ALL streams
                        res = [((names[j] if sc >= THRESHOLD else "Unknown"), float(sc)) for sc, j in zip(best_sc, best_j)]
                    else:
                        res = [("Unknown", 0.0)] * len(jobs)
                    for (s, tid, _), (ident, sc) in zip(jobs, res):
                        trk = s.tracker.tracks[tid]; trk["identity"], trk["score"] = ident, sc
                        if ident == "Unknown":                      # UNKNOWN -> ALERT (once per track)
                            trk["unk_hits"] = trk.get("unk_hits", 0) + 1
                            if trk["unk_hits"] >= ALERT_UNKNOWN_HITS and not trk.get("alerted"):
                                trk["alerted"] = True; s.n_alerts += 1
                                a = dict(video=s.name, track_id=tid,
                                         time_sec=s.processed / s.plan["effective_fps"], score=sc)
                                alerts.append(a)
                                print(f"  ALERT unknown face | {a['video']} | t={a['time_sec']:.1f}s | track {tid} | best score {sc:.2f}")
                        else:
                            trk["unk_hits"] = 0
                    dt = time.perf_counter() - t
                    per_s = {}
                    for s, _, _ in jobs: per_s[id(s)] = per_s.get(id(s), 0) + 1
                    for s, _, _ in groups:
                        if id(s) in per_s:
                            s.t_recog += dt * per_s[id(s)] / len(jobs); s.n_rec += per_s[id(s)]; s.n_rec_batches += 1

                # ---- 5. draw + write each stream's frames IN ORDER (every frame is written) ----
                for s, frames, _ in groups:
                    for i, assigns in enumerate(assigns_by_stream[id(s)]):
                        frame = frames[i]
                        for tid, bbox, _n in assigns:
                            trk = s.tracker.tracks[tid]
                            x1, y1, x2, y2 = map(int, bbox[:4])
                            label, sc = (trk["identity"] or "Unknown"), trk["score"]
                            color = (0, 200, 0) if label != "Unknown" else (0, 0, 255)
                            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                            cv2.putText(frame, f"{label} {sc*100:.0f}%", (x1, max(15, y1 - 8)),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
                        tw = time.perf_counter(); s.writer.write(frame); s.t_write += time.perf_counter() - tw
                    s.tracker.prune()
                    s.processed += len(frames)
            except Exception as e:                       # one bad round must not kill other videos' results
                for s, _, _ in groups: fail(s, f"processing error: {e}")
                continue

            for s in [x for x in active if x.done]: finish(s)

        for fut in encode_futures:                       # drain background encodes
            r = fut.result(); by_name[r["video"]].update(r)
    finally:
        for s in list(active): _close_stream(s)
        encode_pool.shutdown(wait=True)

    order = {os.path.basename(p): i for i, p in enumerate(paths)}
    results.sort(key=lambda r: order.get(r["video"], 1e9))
    total = time.perf_counter() - t_all
    ok = [r for r in results if r["status"] == "Completed"]
    frames_done = sum(r.get("frames_processed", 0) for r in ok)
    print(f"\nConcurrent queue finished -> Total: {len(paths)} | Completed: {len(ok)} | "
          f"Failed: {len(results) - len(ok)} | wall {total:.1f}s | aggregate {frames_done/total:.1f} frames/s "
          f"| unknown alerts: {len(alerts)}")
    return results, alerts



In [ ]:
# ---------- run the queue (kept separate from the definitions so re-running the definitions never reprocesses) ----------
if CONCURRENT_STREAMS > 1 and len(VIDEO_PATHS) > 1:
    print(f"Mode: CONCURRENT ({min(CONCURRENT_STREAMS, len(VIDEO_PATHS))} streams -> 1 shared GPU worker)")
    batch_results, alert_log = process_videos_concurrent(VIDEO_PATHS)
else:
    print("Mode: SEQUENTIAL (single video or CONCURRENT_STREAMS=1)")
    batch_results = process_video_batch(VIDEO_PATHS)
    alert_log = []


## 11) Performance report (Step 10/15 — measured values only)

In [ ]:
def print_batch_report(results, detection_interval, recognition_interval):
    """Every number below comes from measured `results` entries -- nothing is invented."""
    W = 74
    ok = [r for r in results if r["status"] == "Completed"]
    failed = [r for r in results if r["status"] != "Completed"]
    print("=" * W); print("PERFORMANCE REPORT".center(W)); print("=" * W)
    print(f"Videos queued: {len(results)} | Completed: {len(ok)} | Failed: {len(failed)}")
    print(f"Mode settings: DETECTION_INTERVAL={detection_interval}  "
          f"RECOGNITION_INTERVAL={recognition_interval}")
    print(f"GPU: {torch.cuda.get_device_name(0)} | CUDA available: {torch.cuda.is_available()} | "
          f"Backend: TensorRT (no ONNX Runtime in this pipeline)\n")

    for r in ok:
        print(f"--- {r['video']} ---")
        print(f"  Duration               : {r['duration']:.1f}s  "
              f"({r['width']}x{r['height']} @ {r['input_fps']:.1f} input FPS)")

        print(f"  Video decoder          : {r.get('decoder', 'n/a')}")

        encode_time_val = r.get('encode_time')
        total_time_val = r.get('total_time')

        if encode_time_val is not None and total_time_val is not None:
            print(f"  Total processing time  : {r['wall']:.2f}s  "
                  f"(+ {encode_time_val:.2f}s H.264 encode = {total_time_val:.2f}s total)")
        else:
            print(f"  Total processing time  : {r['wall']:.2f}s  "
                  f"(H.264 encode time not available for this processing mode)")

        print(f"  Processing speed       : {r['processing_fps']:.1f} FPS "
              f"({r['speed_ratio']:.2f}x realtime, {'REAL-TIME' if r['real_time'] else 'slower than realtime'})")
        print(f"  Read time              : {r['read']:.2f}s")
        print(f"  Detect time (SCRFD)     : {r['detect']:.2f}s  "
              f"({r['detection_rounds']} of {r['frames_processed']} frames = "
              f"{100*r['detection_rounds']/max(1,r['frames_processed']):.0f}%)")
        print(f"    -> {r['detection_rounds']} detection frames sent in "
              f"{r['detection_batches']} batched TensorRT call(s)")
        print(f"  Track time (IoU)       : {r['track']:.2f}s")
        print(f"  Recognize time (W600K) : {r['recognize']:.2f}s  "
              f"({r['recognition_ops']} recognition ops)")
        print(f"    -> {r['recognition_ops']} recognition ops sent in "
              f"{r['recognition_batches']} batched TensorRT call(s)")
        print(f"  Write time             : {r['write']:.2f}s")
        print(f"  GPU memory allocated   : {r['gpu_mem_mb']:.0f} MB (after this video)")
        print()

    if failed:
        print("Failed videos:")
        for r in failed:
            print(f"  {r['video']}: {r['error']}")
        print()

    if ok:
        tot_wall  = sum(r["wall"] for r in ok)
        tot_frm   = sum(r["frames_processed"] for r in ok)
        tot_det   = sum(r["detection_rounds"] for r in ok)
        tot_rec   = sum(r["recognition_ops"] for r in ok)
        print("-" * W)
        print(f"Aggregate over {len(ok)} completed video(s):")
        print(f"  Total AI processing time : {tot_wall:.2f}s")
        print(f"  Total frames processed   : {tot_frm}")
        print(f"  Total detection rounds   : {tot_det}")
        print(f"  Total recognition ops    : {tot_rec}")
        print(f"  Average processing speed : {tot_frm/tot_wall:.1f} FPS" if tot_wall > 0 else "")
    print("=" * W)


print_batch_report(batch_results, DETECTION_INTERVAL, RECOGNITION_INTERVAL)


## 12) Results per video

In [ ]:
from IPython.display import HTML

# Show the H.264 re-encoded files (the raw mp4v files do not play in the browser).
completed = [r for r in batch_results if r.get("status") == "Completed" and r.get("out_final")]
if not completed:
    print("No finished videos to display.")
tiles = ""
for r in completed:
    with open(r["out_final"], "rb") as fh:
        data = base64.b64encode(fh.read()).decode()
    tiles += (f'<div style="display:inline-block;margin:4px"><b>{r["video"]}</b><br>'
              f'<video width=400 controls src="data:video/mp4;base64,{data}"></video></div>')
HTML(tiles)


In [ ]:
# 1-3 completed videos -> download individually. 4+ -> zip them (fewer browser download prompts).
completed = [r for r in batch_results if r.get("status") == "Completed" and r.get("out_final")]
if not completed:
    print("No finished videos to download.")
elif len(completed) <= 3:
    for r in completed:
        gfiles.download(r["out_final"])
else:
    import zipfile as _zf
    zip_path = "annotated_videos_batch.zip"
    with _zf.ZipFile(zip_path, "w") as zf:
        for r in completed:
            zf.write(r["out_final"])
    print(f"Zipped {len(completed)} videos -> {zip_path}")
    gfiles.download(zip_path)


### Notes
- **GPU only:** face detection and W600K face embeddings run through TensorRT on `cuda:0`.
- **Video decoding:** FFmpeg **NVDEC** is wired into the actual video-processing path. OpenCV is used only for video metadata, image operations, drawing and output writing.
- **Detector:** the notebook uses the official InsightFace `buffalo_l/det_10g.onnx` **SCRFD** detector, converted to a TensorRT engine.
- **Embedder:** the notebook keeps the existing InsightFace `buffalo_l/w600k_r50.onnx` W600K embedding model and 112×112 preprocessing.
- **Gallery database:** existing incremental gallery behavior is preserved. Previously stored embeddings are reused; duplicate uploaded images are skipped.
- **Recognition:** normalized W600K embeddings are compared with the persistent gallery using cosine similarity. `THRESHOLD = 0.60` means 60% or higher is treated as a known match; otherwise the result is Unknown and the alert logic is preserved.
- **Performance pipeline:** the existing frame chunking, TensorRT batching, recognition caching, IoU tracking, multi-video queue, background encoding and performance report are preserved.
- **FPS:** `TARGET_FPS = 30`; the output timeline/FPS logic remains the same. NVDEC decodes source frames and the existing sampling logic decides which frames enter the AI pipeline.
- **Important:** delete any old `s3fd*.engine` / `scrfd_det10g_640_b8.engine` files in the Drive engine folder; the SCRFD engine is now a fixed-batch-1 engine named `scrfd_det10g_640_b1.engine`. Engines are tied to the GPU + TensorRT version: delete them if either changes.
